In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'CHA': ['Coby White'], 'NOP': ['Jordan Hawkins', 'Yves Missi'], 'GSW': ['Kristaps Porziņģis', 'LJ Cryer']}

Out Players:
{'MIN': ['Anthony Edwards', 'Jaden McDaniels'], 'IND': ['Ben Sheppard', 'Aaron Nesmith', 'Pascal Siakam', 'T.J. McConnell', 'Andrew Nembhard'], 'CHI': ['Matas Buzelis', 'Josh Giddey', 'Nick Richards', 'Anfernee Simons'], 'WAS': ['Tre Johnson', 'Alex Sarr', "D'Angelo Russell", 'Trae Young', 'Jaden Hardy', 'Tristan Vukcevic', 'Anthony Davis'], 'MIL': ['Ryan Rollins', 'Giannis Antetokounmpo', 'Gary Trent', 'Bobby Portis', 'Kyle Kuzma', 'Myles Turner'], 'BKN': ['Ziaire Williams', 'Nic Claxton', 'Terance Mann', 'Noah Clowney'], 'MIA': ['Nikola Jović', 'Terry Rozier'], 'TOR': ['Chucky Hepburn'], 'CHA': ['PJ Hall'], 'UTA': ['Ace Bailey', 'Elijah Harkless', 'Isaiah Collier', 'Keyonte George', 'Lauri Markkanen'], 'NOP': ['Dejounte Murray', 'Karlo Matkovic', 'Bryce McGowens', 'Trey Murphy'], 'SAC': ['Russell Westbrook', 'DeMar DeRozan', 'Keegan Murray',

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
72,NaN,2025-26,1642449,Tolu Smith,Tolu,1610612765,DET,Detroit Pistons,22501144,2026-04-06T00:00:00,DET @ ORL,L,13.033333,3,5,0.600,0,0,0.000,2,4,0.5,1,2,3,1,0,0,0,2,3,2,8,8,13.1,0,0,12.0,1,13:02,1,109.9,124.0,124.0,89.4,92.0,92.0,20.5,32.0,32.0,0.100,0.00,12.5,0.077,0.154,0.115,0.0,0.0,0.600,0.592,0.219,0.210,99.29,92.07,76.73,92.07,0.120,25,3.0,5.0,42,85,0.494,10,30,0.333,13,19,0.684,8,27,35,32,21.0,10,7,9,24,19,107,-16.0,100.6,103.9,114.3,118.3,-13.7,-14.4,0.762,1.52,21.6,0.277,0.682,0.473,0.204,0.553,0.573,107.0,103.50,86.25,103,0.428,1610612753,ORL,Orlando Magic,41,81,0.506,11,26,0.423,30,40,0.750,6,29,35,28,15.0,16,9,7,19,24,123,16.0,114.3,118.3,100.6,103.9,13.7,14.4,0.683,1.87,19.2,0.318,0.723,0.527,0.144,0.574,0.624,107.0,103.50,86.25,104,0.572,NaN,PF,25.0
73,NaN,2025-26,1631204,Marcus Sasser,Marcus,1610612765,DET,Detroit Pistons,22501144,2026-04-06T00:00:00,DET @ ORL,L,18.238333,2,8,0.250,1,3,0.333,0,0,0.0,0,2,2,4,2,0,0,2,1,0,5,6,11.4,0,0,12.0,1,18:14,1,94.9,100.0,100.0,86.0,89.2,89.2,8.9,10.8,10.8,0.286,2.00,28.6,0.000,0.105,0.049,14.3,14.3,0.313,0.313,0.213,0.222,104.54,100.01,83.34,100.01,0.036,39,2.0,8.0,42,85,0.494,10,30,0.333,13,19,0.684,8,27,35,32,21.0,10,7,9,24,19,107,-16.0,100.6,103.9,114.3,118.3,-13.7,-14.4,0.762,1.52,21.6,0.277,0.682,0.473,0.204,0.553,0.573,107.0,103.50,86.25,103,0.428,1610612753,ORL,Orlando Magic,41,81,0.506,11,26,0.423,30,40,0.750,6,29,35,28,15.0,16,9,7,19,24,123,16.0,114.3,118.3,100.6,103.9,13.7,14.4,0.683,1.87,19.2,0.318,0.723,0.527,0.144,0.574,0.624,107.0,103.50,86.25,104,0.572,NaN,PG,25.0
74,NaN,2025-26,203501,Tim Hardaway Jr.,Tim,1610612743,DEN,Denver Nuggets,22501147,2026-04-06T00:00:00,DEN vs. POR,W,23.183333,1,7,0.143,1,5,0.200,0,0,0.0,1,0,1,1,1,1,1,1,4,0,3,-24,10.7,0,0,10.0,1,23:11,1,107.7,108.0,108.0,144.0,150.0,150.0,-36.4,-42.0,-42.0,0.056,1.00,11.1,0.042,0.000,0.021,11.1,11.1,0.214,0.214,0.138,0.140,107.99,105.59,87.99,105.59,-0.043,50,1.0,7.0,52,101,0.515,12,38,0.316,21,24,0.875,17,28,45,37,12.0,10,5,5,26,24,137,5.0,128.6,129.2,121.9,125.7,6.7,3.5,0.712,3.08,22.7,0.408,0.627,0.520,0.113,0.574,0.614,97.3,95.55,79.62,106,0.545,1610612757,POR,Portland Trail Blazers,42,89,0.472,25,52,0.481,23,28,0.821,11,28,39,29,18.0,8,5,5,24,26,132,-5.0,121.9,125.7,128.6,129.2,-6.7,-3.5,0.690,1.61,19.1,0.373,0.592,0.480,0.171,0.612,0.651,97.3,95.55,79.62,105,0.455,NaN,SG,33.0
64,NaN,2025-26,1628975,Jevon Cart

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260407_153431.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Chicago Bulls,2026-04-07 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Indiana Pacers,Minnesota Timberwolves,2026-04-07 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Brooklyn Nets,Milwaukee Bucks,2026-04-07 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Toronto Raptors,Miami Heat,2026-04-07 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Boston Celtics,Charlotte Hornets,2026-04-08 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
BOOKMAKER = 'Underdog'
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-07 15:33:37
US latest pull: 2026-04-07 15:34:31


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Collin Sexton,Over,23.5,-137,2026-04-07,2026-04-07T22:32:54Z,2026-04-07 15:33:37
1,PrizePicks,player_points,Collin Sexton,Under,23.5,-137,2026-04-07,2026-04-07T22:32:54Z,2026-04-07 15:33:37
2,PrizePicks,player_points,Tre Jones,Over,18.5,-137,2026-04-07,2026-04-07T22:32:54Z,2026-04-07 15:33:37
3,PrizePicks,player_points,Tre Jones,Under,18.5,-137,2026-04-07,2026-04-07T22:32:54Z,2026-04-07 15:33:37
4,PrizePicks,player_points,Leonard Miller,Over,17.5,-137,2026-04-07,2026-04-07T22:32:54Z,2026-04-07 15:33:37


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Rob Dillingham: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] Ethan Thompson: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Herb Jones: single positional indexer is out-of-bounds
[SKIP] Rob Dillingham: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional 

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Tre Jones,AST,17.52,26.21,32.68,0.1176,0.2245,0.3140,2.06,5.88,10.26,"[0.1734304543877905, 0.1581277672359266, 0.149..."
1,Collin Sexton,AST,19.36,27.84,36.56,0.0398,0.1396,0.2532,0.77,3.89,9.26,"[0.0352112676056338, 0.1685393258426966, 0.061..."
2,Ayo Dosunmu,AST,23.10,32.07,37.58,0.0782,0.1492,0.2384,1.81,4.79,8.96,"[0.0430663221360895, 0.1888574126534466, 0.140..."
3,Bones Hyland,AST,11.83,18.81,26.17,0.0850,0.1423,0.2638,1.01,2.68,6.90,"[0.0, 0.1715265866209262, 0.1920614596670934, ..."
4,Quenton Jackson,AST,13.29,22.44,31.18,0.0594,0.1377,0.2774,0.79,3.09,8.65,"[0.0654664484451718, 0.1504513540621865, 0.162..."
5,Nolan Traore,AST,16.79,26.17,32.80,0.0839,0.1801,0.2980,1.41,4.71,9.78,"[0.0, 0.3621730382293762, 0.1299545159194282, ..."
6,Ben Saraf,AST,14.56,22.36,28.40,0.0651,0.1648,0.2880,0.95,3.69,8.18,"[0.128287363694676, 0.2481389578163771, 0.2814..."
7,Scottie Barnes,AST,26.77,30.67,38.59,0.1066,0.2326,0.3265,2.85,7.13,12.60,"[0.1438021282714984, 0.0557103064066852, 0.112..."
8,Davion Mitchell,AST,16.96,25.94,34.35,0.0919,0.1688,0.2796,1.56,4.38,9.60,"[0.1746724890829694, 0.2597402597402597, 0.164..."
9,Immanuel Quickley,AST,20.40,27.16,33.88,0.0707,0.1514,0.2571,1.44,4.11,8.71,"[0.0904159132007233, 0.2910737386804657, 0.122..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
121,Kevin Durant,REB,5.5,35.94,6.13,0.487,0.512
103,Amen Thompson,REB,8.0,33.49,7.61,0.376,0.529
46,Guerschon Yabusele,REB,7.5,27.75,4.67,0.263,0.737
85,Gary Payton II,REB,5.0,20.17,4.00,0.435,0.448
137,Guerschon Yabusele,PTS,12.5,27.75,11.42,0.530,0.469
111,LaMelo Ball,REB,4.5,31.36,4.58,0.553,0.447
209,Brook Lopez,PTS,10.5,26.65,12.30,0.544,0.456
238,Taurean Prince,PTS,12.5,29.60,9.85,0.388,0.612
159,Tyler Herro,PTS,21.5,34.50,21.40,0.535,0.465
114,Derik Queen,REB,7.5,25.22,7.09,0.391,0.609


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
133,Collin Sexton,PTS,23.5,27.84,19.22,0.371,0.629,PTS,Underdog,Washington Wizards,-6.0,250.0,121.3,30.0,102.38,6.0,-115.0,105.0,0.535,0.488,18.7,18.5,5.06,-3.8,-4.0,0.751,0.226,0.774,-57.75,58.67,0.2,0.2,0.27,0.15,25.48,5.04,0.25,0.04,14.75,4.0
164,Immanuel Quickley,PTS,13.5,27.16,15.59,0.454,0.546,PTS,Underdog,Miami Heat,-2.0,242.0,113.4,11.0,104.40,1.0,-110.0,-105.0,0.524,0.512,13.7,12.5,5.33,0.2,-1.0,-0.038,0.515,0.485,-1.68,-5.31,0.4,0.4,0.60,0.69,31.85,3.80,0.19,0.02,18.67,3.0
8,Davion Mitchell,AST,5.5,25.94,4.38,0.374,0.625,AST,Underdog,Toronto Raptors,2.0,242.0,112.3,8.0,99.32,22.0,-120.0,104.0,0.545,0.490,5.6,5.5,2.01,0.1,0.0,-0.050,0.520,0.480,-4.67,-2.08,0.6,0.5,0.40,0.48,30.53,6.00,0.14,0.03,2.33,3.0
24,Cooper Flagg,AST,5.5,33.88,4.38,0.359,0.641,AST,Underdog,Los Angeles Clippers,12.0,238.5,115.2,18.0,97.22,28.0,105.0,-118.0,0.488,0.541,5.5,6.0,3.06,0.0,0.5,0.000,0.500,0.500,2.50,-7.63,0.2,0.5,0.53,0.38,35.34,4.09,0.30,0.05,3.33,3.0
192,Yves Missi,PTS,8.5,21.96,6.96,0.267,0.734,PTS,Underdog,Utah Jazz,-10.5,243.5,121.2,29.0,103.06,2.0,-114.0,-105.0,0.533,0.512,7.0,6.0,4.71,-1.5,-2.5,0.318,0.375,0.625,-29.61,22.02,0.2,0.2,0.13,0.37,23.27,7.53,0.13,0.03,6.50,2.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
83,Draymond Green,REB,6.5,27.63,6.07,0.341,0.659,REB,PrizePicks,Sacramento Kings,-14.5,234.0,120.4,28.0,100.18,18.0,-137.0,-137.0,0.578,0.578,5.3,5.0,1.89,-0.7,-1.0,0.370,0.356,0.644,-38.41,11.41,0.2,0.2,0.27,0.38,30.41,5.75,0.15,0.05,5.50,4.0
49,Julius Randle,REB,7.5,34.17,7.71,0.438,0.562,REB,PrizePicks,Indiana Pacers,-12.5,233.5,118.3,27.0,101.66,8.0,-137.0,-137.0,0.578,0.578,6.0,6.5,2.45,-1.0,-0.5,0.408,0.342,0.658,-40.84,13.83,0.4,0.3,0.27,0.39,33.39,4.38,0.28,0.05,5.33,3.0
69,Neemias Queta,REB,8.5,26.86,9.58,0.509,0.491,REB,PrizePicks,Charlotte Hornets,-4.5,221.0,113.4,12.0,97.76,26.0,-137.0,-137.0,0.578,0.578,8.3,9.0,2.79,0.3,1.0,-0.108,0.543,0.457,-6.06,-20.94,0.6,0.5,0.60,0.26,28.81,4.61,0.13,0.05,4.33,6.0
143,Bones Hyland,PTS,14.5,18.81,9.96,0.260,0.740,PTS,PrizePicks,Indiana Pacers,-12.5,233.5,118.3,27.0,101.66,8.0,-105.0,-104.0,0.512,0.510,15.1,17.5,6.59,0.6,3.0,-0.091,0.536,0.464,4.65,-8.98,0.4,0.6,0.40,0.15,26.24,3.74,0.22,0.04,2.50,2.0
172,Jayson Tatum,PTS,22.5,35.40,25.81,0.643,0.357,PTS,PrizePicks,Charlotte Hornets,-4.5,221.0,113.4,12.0,97.76,26.0,-108.0,-106.0,0.519,0.515,22.2,23.0,5.35,-0.3,0.5,0.056,0.478,0.522,-7.94,1.45,1.0,0.6,0.47,0.65,33.00,2.60,0.29,0.05,27.25,4.0


In [11]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
208,Klay Thompson,PTS,11.5,26.68,15.14,0.668,0.332,PTS,PrizePicks,Los Angeles Clippers,12.0,238.5,115.2,18.0,97.22,28.0,-105.0,-117.0,0.512,0.539,12.0,11.5,5.56,0.5,0.0,-0.090,0.536,0.464,4.65,-13.94,0.2,0.5,0.47,0.56,21.24,5.53,0.21,0.05,14.57,7.0
198,De'Anthony Melton,PTS,13.5,25.65,11.82,0.404,0.596,PTS,Underdog,Sacramento Kings,-14.5,234.0,120.4,28.0,100.18,18.0,-101.0,-115.0,0.502,0.535,10.3,9.0,8.76,-3.2,-4.5,0.365,0.358,0.642,-28.75,20.03,0.2,0.4,0.47,0.41,24.62,2.71,0.22,0.08,19.00,1.0
76,Kyle Filipowski,REB,9.5,25.36,8.15,0.375,0.625,REB,PrizePicks,New Orleans Pelicans,10.5,243.5,117.4,22.0,100.99,11.0,-119.0,105.0,0.543,0.488,8.5,8.0,3.06,-1.0,-1.5,0.327,0.372,0.628,-31.54,28.74,0.6,0.3,0.40,0.22,27.24,5.34,0.26,0.06,8.40,5.0
65,Tyler Herro,REB,4.5,34.50,4.94,0.460,0.540,REB,PrizePicks,Toronto Raptors,2.0,242.0,112.3,8.0,99.32,22.0,-115.0,102.0,0.535,0.495,4.8,4.5,2.15,0.3,0.0,-0.140,0.556,0.444,3.95,-10.31,0.0,0.5,0.47,0.53,33.97,5.05,0.24,0.03,3.50,4.0
145,Rudy Gobert,PTS,12.5,29.80,12.00,0.451,0.549,PTS,Underdog,Indiana Pacers,-12.5,233.5,118.3,27.0,101.66,8.0,100.0,102.0,0.500,0.495,12.2,13.0,5.07,-0.3,0.5,0.059,0.476,0.524,-4.80,5.85,0.4,0.5,0.40,0.39,30.50,4.07,0.15,0.04,11.00,3.0


### Get top EVs

In [12]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 153  |  Pairs: 423  |  Slate: 8  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [13]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 62  |  Pairs: 128  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
